In [1]:
from pyspark.sql import SparkSession, functions as F

In [2]:
spark = SparkSession.builder \
.appName("Spark SQL") \
.master("local[2]") \
.enableHiveSupport() \
.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/24 09:54:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
! ls -l $SPARK_HOME/jars | grep postgresql

-rw-r--r-- 1 root      root   1221004 Oct 23 18:00 postgresql-42.7.14.jar


In [3]:
! wget -O /opt/examples/datasets/Churn_Modelling.csv \
https://github.com/erkansirin78/datasets/raw/master/Churn_Modelling.csv

--2026-10-24 09:33:31--  https://github.com/erkansirin78/datasets/raw/master/Churn_Modelling.csv
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/erkansirin78/datasets/master/Churn_Modelling.csv [following]
--2026-10-24 09:33:32--  https://raw.githubusercontent.com/erkansirin78/datasets/master/Churn_Modelling.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 674856 (659K) [text/plain]
Saving to: ‘/opt/examples/datasets/Churn_Modelling.csv’

/opt/examples/datas 100%[===================>] 659.04K  --.-KB/s    in 0.1s    

2026-10-24 09:33:32 (5.30 MB/s) - ‘/opt/examples/datasets/Churn_Modelling.c

In [4]:
df = spark.read.format("csv") \
.option("header", True) \
.option("inferSchema", True) \
.option("sep", ",") \
.load("file:///opt/examples/datasets/Churn_Modelling.csv")

In [5]:
df.limit(5).toPandas()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
df.createOrReplaceTempView("churn")

In [10]:
df2 = spark.sql("""

select * from churn
where Geography = 'France'

""")

In [12]:
type(df2)

pyspark.sql.classic.dataframe.DataFrame

In [13]:
df_spain = spark.sql("""
select * from churn
where Geography = 'Spain'

""")

In [14]:
df_spain.show()

+---------+----------+---------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|RowNumber|CustomerId|  Surname|CreditScore|Geography|Gender|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited|
+---------+----------+---------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|        2|  15647311|     Hill|        608|    Spain|Female| 41|     1| 83807.86|            1|        0|             1|      112542.58|     0|
|        5|  15737888| Mitchell|        850|    Spain|Female| 43|     2|125510.82|            1|        1|             1|        79084.1|     0|
|        6|  15574012|      Chu|        645|    Spain|  Male| 44|     8|113755.78|            2|        1|             0|      149756.71|     1|
|       12|  15737173|  Andrews|        497|    Spain|  Male| 24|     3|      0.0|            2|        1|             0|       76

In [16]:
spark.sql("""

select Geography, count(*) as count from churn
group by Geography

""").show()

+---------+-----+
|Geography|count|
+---------+-----+
|  Germany| 2509|
|   France| 5014|
|    Spain| 2477|
+---------+-----+



In [17]:
df.printSchema()

root
 |-- RowNumber: integer (nullable = true)
 |-- CustomerId: integer (nullable = true)
 |-- Surname: string (nullable = true)
 |-- CreditScore: integer (nullable = true)
 |-- Geography: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Tenure: integer (nullable = true)
 |-- Balance: double (nullable = true)
 |-- NumOfProducts: integer (nullable = true)
 |-- HasCrCard: integer (nullable = true)
 |-- IsActiveMember: integer (nullable = true)
 |-- EstimatedSalary: double (nullable = true)
 |-- Exited: integer (nullable = true)



In [27]:
gold = (15647311, 15589475, 15684171)

In [28]:
spark.sql(f"""

select * from churn
where CustomerId in {gold}

""").show()

+---------+----------+-------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|RowNumber|CustomerId|Surname|CreditScore|Geography|Gender|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited|
+---------+----------+-------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|        2|  15647311|   Hill|        608|    Spain|Female| 41|     1| 83807.86|            1|        0|             1|      112542.58|     0|
|       31|  15589475|Azikiwe|        591|    Spain|Female| 39|     3|      0.0|            3|        1|             0|      140469.38|     1|
|       45|  15684171|Bianchi|        660|    Spain|Female| 61|     5|155931.11|            1|        1|             1|      158338.39|     0|
+---------+----------+-------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+

In [29]:
spark.stop()